In [1]:
import gc
# import torch

gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

import os
os.listdir(); os.chdir("/aiau010_scratch/azm0269/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()

In [86]:
import json
from pathlib import Path
import pandas as pd


root = Path("/aiau010_scratch/azm0269/clover/outputs/") 
eval_folder = "evals"
manifest_file = "eval_manifest.json"
BASELINES = ["ddpo", "md3po"] #, 

def combine_eval_manifests():
    combined_eval_manifests = pd.DataFrame() 
    for baseline in BASELINES:
        eval_path = root / baseline / eval_folder / "images" / manifest_file
        data = json.load(open(eval_path))
        df = pd.DataFrame(data)
        df['baseline'] = baseline
        df['epoch'] = df['image'].str.split('epoch_').str[1].str.split('_').str[0].astype(int)
        combined_eval_manifests = pd.concat([combined_eval_manifests, df], ignore_index=False)
    return combined_eval_manifests

def get_eval_images_by_baseline_by_prompt(combined_eval_manifests, baseline="ddpo"):
    same_image_per_prompt = []
    prompts = combined_eval_manifests['prompt'].unique().tolist()
    for prompt in prompts:
        prompt_images = combined_eval_manifests[(combined_eval_manifests['prompt'] == prompt) & (combined_eval_manifests['baseline'] == baseline)]['image'].tolist()
        same_image_per_prompt.append({
            'prompt': prompt,
            'baseline': baseline,
            'images': prompt_images
        })
    return same_image_per_prompt

def plot_eval_images_by_baseline_by_prompt(same_image_per_prompt, skip=4):
    import matplotlib.pyplot as plt
    from PIL import Image

    n_rows = len(same_image_per_prompt)
    n_cols = len(same_image_per_prompt[0]['images'])

    fig, axes = plt.subplots(
        n_rows,
        n_cols//skip,
        figsize=(4 * (n_cols//skip), 4 * n_rows),
        squeeze=False
    )

    for row, row_data in enumerate(same_image_per_prompt):
        for col, image_path in enumerate(row_data['images']):
            if col % skip == 0:
                # continue
                img = Image.open(image_path)
                col_num = col // skip - 1
                axes[row, col_num].imshow(img)
                axes[row, col_num].axis("off")
                axes[row, col_num].set_title(row_data['baseline'])
    plt.tight_layout()
    plt.show()

In [ ]:
combined_eval_manifests = combine_eval_manifests()
eval_images_by_baseline_by_prompt = get_eval_images_by_baseline_by_prompt(combined_eval_manifests, baseline="md3po")
# eval_images_by_baseline_by_prompt
plot_eval_images_by_baseline_by_prompt(eval_images_by_baseline_by_prompt, skip=5)